### Optional Module

If you do not already have the **ipympl** module, you can install it with

> conda install ipympl

This makes the plots into interactive figures. Activate it with following cell.

If you are using pip instead of conda package manager, you can use

> pip install ipympl



In [ ]:
%matplotlib widget

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Section 1 - Simple Filtering

In [ ]:
n_points = 1000
time = np.arange(n_points)
sine_period = 50  # seconds
sine_freq = 1 / sine_period
exp_tau = 1       # time constant of exponential
n_pulses = 10

# Baseline ripple
sin_amp = 0.25
sine_wave = sin_amp*np.sin(2 * np.pi * time / sine_period)
timestream = sine_wave.copy()

# Inject 10 falling exponential pulses
for _ in range(n_pulses):
    pulse_start = np.random.randint(0, n_points - 100)  # Leave buffer at end
    pulse_length = 20  # long enough for decay
    decay = np.exp(-np.arange(pulse_length) / exp_tau)
    timestream[pulse_start:pulse_start + pulse_length] += decay

plt.figure(figsize=(8, 4))
plt.plot(time, timestream, label="Timestream")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
# plt.title("Synthetic Timestream with Sine Wave and Exponential Pulses")
# plt.grid(True)
plt.legend()
plt.show()


Change the variable "filter_class" for the different filter profiles.

In [ ]:
filt_class = "Band" # Low, High, Bandpass

fft_data = np.fft.fft(timestream)
freq = np.fft.fftfreq(n_points, d=1.0)  # sampling rate = 1 Hz

cutoff1 = 0.015  # Hz — slightly above 0.02 Hz sine
cutoff2 = 0.025  # Hz — slightly above 0.02 Hz sine
high_pass_mask = np.abs(freq) < cutoff2
low_pass_mask = np.abs(freq) > cutoff2

if filt_class == "Low":
    fft_filtered = fft_data * low_pass_mask
elif filt_class == "High":
    fft_filtered = fft_data * high_pass_mask
elif filt_class == "Band":
    fft_filtered = fft_data * ( np.abs(freq) > cutoff1 )  * ( np.abs(freq) < cutoff2 )
    

# Inverse FFT to get filtered timestream
filtered_timestream = np.fft.ifft(fft_filtered).real


# Plot frequency spectrum
plt.figure(figsize=(8, 4))
plt.plot(freq[:n_points//2], np.abs(fft_data[:n_points//2]), label="Original Spectrum")
plt.plot(freq[:n_points//2], np.abs(fft_filtered[:n_points//2]), label=filt_class+" Pass Filtered", alpha=0.7)
# plt.axvline(x=sine_freq, color='r', linestyle='--', label='Sine Freq (0.02 Hz)')
plt.xlabel("Frequency (Hz)")
plt.ylabel("Magnitude")
plt.title("FFT: Before and After "+filt_class+ " Pass Filtering")
plt.grid(True)
plt.xscale('log')
plt.legend()
plt.tight_layout()
plt.show()

# Plot timestreams
plt.figure(figsize=(8, 4))
plt.plot(time, timestream, label="Original Timestream", alpha=0.5)
plt.plot(time, timestream-filtered_timestream, label=filt_class+" Pass Filtered", linewidth=2)
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title(f"Timestream: {filt_class} Pass Filtered")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Section 2 - Matched Filtering

In [ ]:
from scipy.signal import convolve

In [ ]:
# Matched filter
n_points = 1000
time = np.arange(n_points)
sine_period = 5000  # seconds
sine_freq = 1 / sine_period
exp_tau = 3      # time constant of exponential
n_pulses = 10

# Base sine wave
sin_amp = 0.001
sine_wave = sin_amp*np.sin(2 * np.pi * time / sine_period)

mean = 0.0
std_dev = 0.1 
noise = np.random.normal(loc=mean, scale=std_dev, size=n_points)

timestream = sine_wave.copy() + noise

# Inject 10 pulses
for _ in range(n_pulses):
    pulse_start = np.random.randint(0, n_points - 100)  # Leave buffer at end
    pulse_length = 20  # long enough for decay
    decay = np.exp(-np.arange(pulse_length) / exp_tau)
    timestream[pulse_start:pulse_start + pulse_length] += decay

plt.figure(figsize=(8, 4))
plt.plot(time, timestream, label="Timestream")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.show()

In [ ]:
template_pulse = np.array([0,0,0]+list(decay))

plt.figure()
plt.plot(range(len(decay)+3), [0,0,0]+list(decay))
plt.show()

In [ ]:
# Reverse the template in time for matched filtering
template_flipped = template_pulse[::-1]

# Apply matched filter via convolution
filtered_output = convolve(timestream, template_flipped, mode='same')  # or 'valid'

In [ ]:
plt.figure()
plt.plot(range(1000),timestream,label = "Original")
plt.plot(range(1000),filtered_output+2,label = "Matched Filter (+ Offset) Output")
plt.legend()
plt.show()

## Section 3 - Deconvolution

In [ ]:
from scipy.signal import fftconvolve

In [ ]:
# True signal (strong peaks)
n = 200
true_signal = np.zeros(n)
true_signal[50] = 6.0
true_signal[100] = 5.0
true_signal[150] = 7.0

# Gaussian PSF in 1D
def gaussian_psf(length=20, sigma=3.0):
    x = np.arange(length) - length // 2
    psf = np.exp(-x**2 / (2 * sigma**2))
    return 10*psf / psf.sum()

# Richardson–Lucy Deconvolution (1D)
def richardson_lucy_1d(observed, psf, iterations=500):
    eps = 1e-8
    estimate = np.ones_like(observed)
    psf_mirror = psf[::-1]

    for _ in range(iterations):
        conv = fftconvolve(estimate, psf, mode='same') + eps
        ratio = observed / conv
        correction = fftconvolve(ratio, psf_mirror, mode='same')
        estimate *= correction
    return estimate

In [ ]:
psf = gaussian_psf()

plt.figure()
plt.plot(psf)
plt.title("Guassian PSF")
plt.show()

In [ ]:
# Create initial guess by bluring and adding noise
blurred = fftconvolve(true_signal, psf, mode='same')
np.random.seed(42)
noise_level = 0.5
observed = blurred + np.random.normal(0, noise_level, size=blurred.shape)

restored = richardson_lucy_1d(observed, psf, iterations=500)


fig, axs = plt.subplots(3, 1, figsize=(6, 8), sharex=False)

axs[0].plot(true_signal, color='green')
axs[0].set_title("True Signal")


axs[1].plot(observed, color='orange',label="Observed")
axs[1].plot(true_signal, color='green',label="True Signal")
axs[1].legend()

axs[2].plot(restored, color='red',label="Deconvolved")
axs[2].set_title("Deconvolved")

plt.tight_layout()
plt.show()



### Exercise - 2D Deconvolution
The module "skimage" has a 2D Richardson-Lucy method. Can you make your own version, following the outline of "richardson_lucy_1d"?

Compare how well both work by importing your own image, blurring it by convolution with the provided 2D PSF, and then deconvolving it with your method and the skimage method. You can import your image using

> from skimage import io
> 
> img = io.imread('file_path')

If you do not have this module yet, install with

> conda install scikit-image

or if you are not using conda, then use

> pip install scikit-image

In [ ]:
from skimage.restoration import richardson_lucy
from scipy.fft import fft2, ifft2, fftshift
from skimage import io

In [ ]:
# 2D Gaussian PSF
def gaussian_psf(size=30, sigma=10):
    ax = np.arange(-size // 2 + 1., size // 2 + 1.)
    xx, yy = np.meshgrid(ax, ax)
    psf = np.exp(-(xx**2 + yy**2) / (2. * sigma**2))
    return psf / np.sum(psf)

In [ ]:
# Import your image and blur it with the PSF.
# Adjust the blue with the "size" and "sigma" parameters if your image is too blurry or not blurry enough.

#img = io.imread('file_path')

In [ ]:
# Deconvolve with the skimage.restoration function and display the recovered image.

In [ ]:
# Make your 2D Richard-Lucy Function using the imported 2D fft functions

In [ ]:
# Deconvolve with your function and display the recovered image.
